In [13]:
import pandas as pd
from typing import List, Set
import xml.etree.ElementTree as ET

### Drug Interaction Action Categories

These lists are used to classify the interaction mechanisms extracted from the DrugBank XML file into standardized groups (`activator`, `inhibitor`, or `neutral`).

#### Activators & Agonists
Includes all variations of activation, positive modulation, agonism, and functional stimulation:
* `activator`, `agonist`, `partial agonist`
* `positive allosteric modulator`, `positive modulator`
* `potentiator`, `inducer`, `upregulator`
* `stimulator`, `stabilization`

#### Inhibitors & Antagonists
Includes all variations of inhibition, negative modulation, antagonism, and blocking mechanisms:
* `inhibitor`, `weak inhibitor`, `antagonist`, `partial antagonist`
* `inverse agonist`, `downregulator`, `suppressor`, `reducer`
* `inhibitory allosteric modulator`, `negative modulator`
* `nucleotide exchange blocker`, `inactivator`, `blocker`
* `inhibition of synthesis`, `inhibits downstream inflammation cascades`
* `translocation inhibitor`

#### Neutral & Binding Actions
Includes carriers, binders, structural components, and unclassified or general biological interactions:
* `adduct`, `allosteric modulator`, `antibody`, `antisense oligonucleotide`
* `binder`, `binding`, `carrier`, `chaperone`, `chelator`, `cleavage`
* `cofactor`, `component of`, `cross-linking/alkylation`, `degradation`
* `disruptor`, `gene replacement`, `incorporation into and destabilization`
* `intercalation`, `ligand`, `metabolizer`, `modulator`, `multitarget`
* `neutralizer`, `other`, `other/unknown`, `oxidizer`, `product of`
* `protector`, `regulator`, `substrate`, `transporter`, `unknown`

In [14]:
ACTIVATOR = [
    'activator', 'agonist', 'partial agonist', 'positive allosteric modulator',
    'positive modulator', 'potentiator', 'inducer', 'upregulator', 
    'stimulator', 'stabilization'
] 

INHIBITOR = [
    'inhibitor', 'weak inhibitor', 'antagonist', 'partial antagonist',
    'inverse agonist', 'downregulator', 'suppressor', 'reducer',
    'inhibitory allosteric modulator', 'negative modulator', 
    'nucleotide exchange blocker', 'inactivator', 'blocker', 
    'inhibition of synthesis', 'inhibits downstream inflammation cascades',
    'translocation inhibitor'
]

NEUTRAL = [
    'adduct', 'allosteric modulator', 'antibody', 'antisense oligonucleotide',
    'binder', 'binding', 'carrier', 'chaperone', 'chelator', 'cleavage',
    'cofactor', 'component of', 'cross-linking/alkylation', 'degradation',
    'disruptor', 'gene replacement', 'incorporation into and destabilization',
    'intercalation', 'ligand', 'metabolizer', 'modulator', 'multitarget',
    'neutralizer', 'other', 'other/unknown', 'oxidizer', 'product of',
    'protector', 'regulator', 'substrate', 'transporter', 'unknown'
]

In [15]:
def parsing(xml_path: str, dict_dfs: dict, drugs: List[str], organisms: List[str]) -> dict:
    """
    Parses the input DrugBank XML file and extracts targets, enzymes, carriers, 
    and transporters data into a dictionary of lists. Each list will later 
    be converted into a pandas DataFrame.
    """
    print("Parsing XML file...")
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    ns = {'d': 'http://www.drugbank.ca'}

    # Iterate over each <drug> element in the XML catalog
    for drug in root.findall('d:drug', ns):
        
        # Extract primary DrugBank ID
        drug_id_elem = drug.find("d:drugbank-id[@primary='true']", ns)
        drug_id = drug_id_elem.text if drug_id_elem is not None else None

        # Extract drug name and normalize to lowercase for matching
        drug_name_elem = drug.find("d:name", ns)
        drug_name = drug_name_elem.text if drug_name_elem is not None else None
        
        if not drug_name:
            continue
            
        drug_name_lower = drug_name.lower()

        # Process only drugs specified in the input list
        if drug_name_lower in drugs:
            
            # Store base drug information
            dict_dfs["drugs_df"].append({
                "Drug_ID": drug_id,
                "Drug_name": drug_name
            })

            # =========================================================================
            # 1. TARGETS & DRUG-TARGET INTERACTIONS
            # =========================================================================
            for target in drug.findall("d:targets/d:target", ns):
                target_id = target.findtext("d:id", namespaces=ns)
                target_name = target.findtext("d:name", namespaces=ns)
                target_organism = target.findtext("d:organism", namespaces=ns)
                
                if target_organism not in organisms:
                    continue

                # Classify interactions into activator or inhibitor groups
                t_act = [a.text for a in target.findall("d:actions/d:action", ns) if a.text in ACTIVATOR]
                t_inh = [a.text for a in target.findall("d:actions/d:action", ns) if a.text in INHIBITOR]
                
                target_actions = (["activator"] if t_act else []) + (["inhibitor"] if t_inh else [])

                # Extract polypeptide details (gene name and UniProt ID)
                polypeptide = target.find("d:polypeptide", ns)
                if polypeptide is not None:
                    gene_name = polypeptide.findtext("d:gene-name", namespaces=ns)
                    uniprot_id = polypeptide.findtext(
                        "d:external-identifiers/d:external-identifier[d:resource='UniProtKB']/d:identifier", 
                        namespaces=ns
                    )
                else:
                    gene_name = uniprot_id = None

                # Append to target and interaction dictionaries
                dict_dfs["targets_df"].append({
                    "Target_ID": target_id,
                    "Target_name": target_name,
                    "Polypeptide_gene_name": gene_name,
                    "Target_UniProtKB": uniprot_id,
                })

                dict_dfs["drug_targets_df"].append({
                    "Drug_ID": drug_id,
                    "Drug_name": drug_name,
                    "Inter_ID": target_id,
                    "Inter_name": target_name,
                    "Polypeptide_gene_name": gene_name,
                    "Inter_UniProtKB": uniprot_id,
                    "Actions": target_actions,
                    "Inter_type": "Target"
                })

            # =========================================================================
            # 2. ENZYMES & DRUG-ENZYME INTERACTIONS
            # =========================================================================
            for enzyme in drug.findall("d:enzymes/d:enzyme", ns):
                enzyme_id = enzyme.findtext("d:id", namespaces=ns)
                enzyme_name = enzyme.findtext("d:name", namespaces=ns)
                enzyme_organism = enzyme.findtext("d:organism", namespaces=ns)
                
                if enzyme_organism not in organisms:
                    continue

                e_act = [a.text for a in enzyme.findall("d:actions/d:action", ns) if a.text in ACTIVATOR]
                e_inh = [a.text for a in enzyme.findall("d:actions/d:action", ns) if a.text in INHIBITOR]
                
                enzyme_actions = (["activator"] if e_act else []) + (["inhibitor"] if e_inh else [])

                polypeptide = enzyme.find("d:polypeptide", ns)
                gene_name = polypeptide.findtext("d:gene-name", namespaces=ns) if polypeptide is not None else None
                uniprot_id = polypeptide.findtext("d:external-identifiers/d:external-identifier[d:resource='UniProtKB']/d:identifier", namespaces=ns) if polypeptide is not None else None

                dict_dfs["enzymes_df"].append({
                    "Enzyme_ID": enzyme_id,
                    "Enzyme_name": enzyme_name,
                    "Polypeptide_gene_name": gene_name,
                    "Enzyme_UniProtKB": uniprot_id,
                })

                dict_dfs["drug_enzymes_df"].append({
                    "Drug_ID": drug_id,
                    "Drug_name": drug_name,
                    "Inter_ID": enzyme_id,
                    "Inter_name": enzyme_name,
                    "Polypeptide_gene_name": gene_name,
                    "Inter_UniProtKB": uniprot_id,
                    "Actions": enzyme_actions,
                    "Inter_type": "Enzyme"
                })

            # =========================================================================
            # 3. CARRIERS & DRUG-CARRIER INTERACTIONS
            # =========================================================================
            for carrier in drug.findall("d:carriers/d:carrier", ns):
                carrier_id = carrier.findtext("d:id", namespaces=ns)
                carrier_name = carrier.findtext("d:name", namespaces=ns)
                carrier_organism = carrier.findtext("d:organism", namespaces=ns)
                
                if carrier_organism not in organisms:
                    continue

                c_act = [a.text for a in carrier.findall("d:actions/d:action", ns) if a.text in ACTIVATOR]
                c_inh = [a.text for a in carrier.findall("d:actions/d:action", ns) if a.text in INHIBITOR]
                
                carrier_actions = (["activator"] if c_act else []) + (["inhibitor"] if c_inh else [])

                polypeptide = carrier.find("d:polypeptide", ns)
                gene_name = polypeptide.findtext("d:gene-name", namespaces=ns) if polypeptide is not None else None
                uniprot_id = polypeptide.findtext("d:external-identifiers/d:external-identifier[d:resource='UniProtKB']/d:identifier", namespaces=ns) if polypeptide is not None else None

                dict_dfs["carriers_df"].append({
                    "Carrier_ID": carrier_id,
                    "Carrier_name": carrier_name,
                    "Polypeptide_gene_name": gene_name,
                    "Carrier_UniProtKB": uniprot_id,
                })

                dict_dfs["drug_carriers_df"].append({
                    "Drug_ID": drug_id,
                    "Drug_name": drug_name,
                    "Inter_ID": carrier_id,
                    "Inter_name": carrier_name,
                    "Polypeptide_gene_name": gene_name,
                    "Inter_UniProtKB": uniprot_id,
                    "Actions": carrier_actions,
                    "Inter_type": "Carrier"
                })

            # =========================================================================
            # 4. TRANSPORTERS & DRUG-TRANSPORTER INTERACTIONS
            # =========================================================================
            for transporter in drug.findall("d:transporters/d:transporter", ns):
                transporter_id = transporter.findtext("d:id", namespaces=ns)
                transporter_name = transporter.findtext("d:name", namespaces=ns)
                transporter_organism = transporter.findtext("d:organism", namespaces=ns)
                
                if transporter_organism not in organisms:
                    continue

                tr_act = [a.text for a in transporter.findall("d:actions/d:action", ns) if a.text in ACTIVATOR]
                tr_inh = [a.text for a in transporter.findall("d:actions/d:action", ns) if a.text in INHIBITOR]
                
                transporter_actions = (["activator"] if tr_act else []) + (["inhibitor"] if tr_inh else [])

                polypeptide = transporter.find("d:polypeptide", ns)
                gene_name = polypeptide.findtext("d:gene-name", namespaces=ns) if polypeptide is not None else None
                uniprot_id = polypeptide.findtext("d:external-identifiers/d:external-identifier[d:resource='UniProtKB']/d:identifier", namespaces=ns) if polypeptide is not None else None

                dict_dfs["transporters_df"].append({
                    "Transporter_ID": transporter_id,
                    "Transporter_name": transporter_name,
                    "Polypeptide_gene_name": gene_name,
                    "Transporter_UniProtKB": uniprot_id,
                })

                dict_dfs["drug_transporters_df"].append({
                    "Drug_ID": drug_id,
                    "Drug_name": drug_name,
                    "Inter_ID": transporter_id,
                    "Inter_name": transporter_name,
                    "Polypeptide_gene_name": gene_name,
                    "Inter_UniProtKB": uniprot_id,
                    "Actions": transporter_actions,
                    "Inter_type": "Transporter"
                })

## Input & Output Parameters

Specify the input tissue context, organisms of intereset, and define the necessary file paths for the analysis pipeline:

* **Input Tissue**: The biological tissue context (e.g., `"skin"` when working with `MEL`).
* **DrugBank XML**: Path to the source DrugBank database file.
* **Output Directory**: Target path where results, tables, and figures will be saved.
* **Drug List**: Path to the file containing the list of drugs for target retrieval.
* **Organisms**: List of organisms used to filter protein targets, enzymes, carriers, and transporters extracted from DrugBank.

In [16]:
tissue = "skin"
tissue = tissue.replace(" ", "_")
drugbank_path = "./full_database.xml"
output_folder = "./Dataframes"
drug_path = f"./drugs_{tissue}.csv"
organisms = ["Humans"]

### Data Structures & Pipeline Initialization

This section initializes the empty lists and master dictionary used to store the parsed records, loads the target drug list from disk, sets the target organism filter, and finally invokes the `parsing()` function on the DrugBank XML database.

* **`dict_dfs`**: A dictionary container holding lists for drugs, target proteins, enzymes, carriers, transporters, and their respective drug-interaction mappings.
* **`drugs`**: A list of drug names loaded from the specified file path (extracted from column `.x`).
* **`organisms`**: Set to `['Humans']` to restrict interactions strictly to human protein targets.
* **`parsing()`**: Executes the XML extraction pipeline using the parameters defined above.

In [18]:
# -------------------- <Data lists and dict> -------------------- 
drugs_df = []
drug_drugs_df = []
targets_df = []
drug_targets_df = []
enzymes_df = []
drug_enzymes_df = []
carriers_df = []
drug_carriers_df = []
transporters_df = []
drug_transporters_df = []

dict_dfs = {
    "drugs_df": drugs_df,
    "drug_drugs_df": drug_drugs_df,
    "targets_df": targets_df,
    "drug_targets_df": drug_targets_df,
    "enzymes_df": enzymes_df,
    "drug_enzymes_df": drug_enzymes_df,
    "carriers_df": carriers_df,
    "drug_carriers_df": drug_carriers_df,
    "transporters_df": transporters_df,
    "drug_transporters_df": drug_transporters_df
    }

drugs = pd.read_csv(drug_path)
drugs = list(drugs.x)
organisms = ['Humans']

parsing(drugbank_path, dict_dfs, drugs, organisms)

Parsing XML file...


### Data Processing, Filtering & Export

Once the raw records are parsed into the `dict_dfs` structure, this processing pipeline transforms them into structured Pandas DataFrames, cleans the interactions, handles duplicates, and exports the final dataset.

* **DataFrame Conversion**: Converts raw list collections into individual DataFrames and merges all interaction types (targets, enzymes, carriers, transporters) into a unified master table.
* **Interaction Filtering**: Removes ambiguous or missing interaction types (retaining only clean, single-action mappings) and unpacks the action list.
* **Gene Quality Control**: Filters out rows with missing or empty polypeptide gene names.
* **Consistency Check & Deduplication**: Detects and discards duplicate drug-gene pairs with conflicting action types, then collapses unique interactions per drug into semicolon-separated strings.
* **Export**: Saves the final processed interaction table as a CSV file named dynamically using the target tissue context (e.g., `targets_skin_interactions.csv`).

In [ ]:
# Convert raw list collections into DataFrames and merge all interaction types
drugs_df = pd.DataFrame(dict_dfs["drugs_df"])
targets_df = pd.DataFrame(dict_dfs["targets_df"])
drug_target_df = pd.DataFrame(dict_dfs["drug_targets_df"])
drug_enzyme_df = pd.DataFrame(dict_dfs["drug_enzymes_df"])
drug_carrier_df = pd.DataFrame(dict_dfs["drug_carriers_df"])
drug_transporter_df = pd.DataFrame(dict_dfs["drug_transporters_df"])

drug_interactions_df = pd.concat([drug_target_df, drug_enzyme_df, drug_carrier_df, drug_transporter_df], axis = 0)

# Interactions for ReCAST, no interactions for scDrugLink
interactions = True
if interactions:
    # Filter out interactions with either no action (empty) or multiple conflicting action types
    empty_mask = (drug_interactions_df['Actions'].str.len() == 0)
    ambigous_mask = (drug_interactions_df['Actions'].str.len() == 2)
    rmvd_mask = empty_mask | ambigous_mask
    drug_interactions_filtered_df = drug_interactions_df[~rmvd_mask].copy()
    drug_interactions_filtered_df['Actions'] = drug_interactions_filtered_df['Actions'].apply(
        lambda x: x[0]
    )

    # Remove rows where the gene name is missing, NaN, or empty whitespace
    cleaned_df = drug_interactions_filtered_df[
        ~drug_interactions_filtered_df["Polypeptide_gene_name"].isna() & 
        (drug_interactions_filtered_df["Polypeptide_gene_name"].astype(str).str.strip() != "")
    ]

    # Check for duplicated drug-gene interactions, ensuring that action types are consistent
    subset_cols = ["Drug_name", "Polypeptide_gene_name"]
    duplicate_mask = cleaned_df.duplicated(subset=subset_cols, keep=False)
    df_duplicates = cleaned_df[duplicate_mask]

    action_check = df_duplicates.groupby(subset_cols).agg(
        unique_actions=('Actions', 'nunique'),
        first_action=('Actions', 'first')
    ).reset_index()

    # Identify groups with inconsistent actions and remove them from the dataset
    inconsistent_actions_groups = action_check[action_check['unique_actions'] > 1]
    if len(inconsistent_actions_groups) != 0:
        print("Found drug-target interactions inconsistent: removing them...")
        print(inconsistent_actions_groups[subset_cols])
        inconsistent_keys = inconsistent_actions_groups[subset_cols]
        df_temp = cleaned_df.merge(
                    inconsistent_keys, 
                    on=subset_cols, 
                    how='left', 
                    indicator=True
                    )
        cleaned_df = df_temp[df_temp['_merge'] == 'left_only'].drop(columns=['_merge'])


    # Remove remaining duplicated drug-gene interactions keeping the first occurrence
    subset_cols = ["Drug_name", "Polypeptide_gene_name"]
    cleaned_df_unique = cleaned_df.drop_duplicates(subset=subset_cols, keep='first')
else:
    cleaned_df_unique = drug_interactions_df

# Group by drug name and collapse multiple targets and actions into semicolon-separated strings
df_grouped = cleaned_df_unique.groupby("Drug_name", as_index=False).agg({
    "Polypeptide_gene_name": lambda x: ";".join(filter(None, map(str, x))),
    "Actions": lambda x: ";".join(filter(None, map(str, x))),
})
df_grouped.rename(columns={"Drug_name": "drug_name", "Polypeptide_gene_name": "gene_names", "Actions": "actions"}, inplace=True)

# Export final processed interactions to CSV
if interactions:
    df_grouped.to_csv(f"drug_targets_moa_{tissue}.csv", index=False)
else:
    df_grouped.drop(columns = "actions", inplace=True)
    df_grouped.to_csv(f"drug_targets_{tissue}.csv", index=False)